# Local renewable inputs and model-ready preparation

The GFS f3/f6 input CSVs are supplied in the repository. The original hourly reconstruction, PV/wind availability calculations, demand generation and price/signal assembly are retained. Run this notebook after notebook 02, or set `USE_NEW_SIGNALS=False` to use supplied signals. Model-ready outputs are written to `generated/upstream/model_ready/`. The optimization horizon is selected in the principal notebook.


In [ ]:
from pathlib import Path
import os

# Open from the repository root or its notebooks/ subfolder.
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "Csv files of input and output").is_dir():
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "Csv files of input and output").is_dir():
    raise FileNotFoundError("Set REPO_ROOT to the downloaded repository folder")
DATA = REPO_ROOT / "Csv files of input and output"
WORK = REPO_ROOT / "generated/upstream"
WORK.mkdir(parents=True, exist_ok=True)

USE_NEW_SIGNALS = True  # False uses the archived model-ready emission signals.
GFS_SOURCE = DATA / "results/ML/GFS for local RES"
GFS_WORK = WORK / "gfs"
GFS_WORK.mkdir(parents=True, exist_ok=True)
SIGNAL_SOURCE = WORK / "signals" if USE_NEW_SIGNALS else DATA / "optimization input-baseline"
PRICE_SOURCE = DATA / "optimization input-baseline/grid_price_hourly_2024.csv"
MODEL_READY_OUTPUT = WORK / "model_ready"


# 3. GFS-based local renewable input preparation

This section keeps the GFS organizing cells used before the optimization utility cell. It prepares hourly 2024 weather inputs for PV and wind availability calculations.


## 3.1 T2M hourly reconstruction


In [ ]:
import pandas as pd
import numpy as np
# Clean GFS T2M f3/f6 instantaneous forecasts, merge them into a complete 3-hourly series,
# and generate a smooth hourly T2M timeseries for 2024 via linear interpolation.

# ============================
# CONFIG – file paths
# ============================
PATH_T2M_F3 = str(GFS_SOURCE / "t2m/t2m_f3_2024.csv")
PATH_T2M_F6 = str(GFS_SOURCE / "t2m/t2m_f6_2024.csv")

OUT_T2M_3H = str(GFS_WORK / "t2m_3hour_2024_combined.csv")
OUT_T2M_1H = str(GFS_WORK / "t2m_hourly_2024_from_f3f6.csv")

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 valid times: 03, 09, 15, 21 (4 per day)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 valid times: 00, 06, 12, 18 (4 per day)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# ============================
# Helper: fix missing by previous day and report
# ============================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# ============================
# 1) Read raw T2M f3 and f6 files
# ============================
t3 = pd.read_csv(PATH_T2M_F3)
t6 = pd.read_csv(PATH_T2M_F6)

# Build datetime_utc from "day" + "valid_time"
t3["datetime_utc"] = pd.to_datetime(
    t3["day"].astype(str) + " " + t3["valid_time"].astype(str),
    utc=True
)
t6["datetime_utc"] = pd.to_datetime(
    t6["day"].astype(str) + " " + t6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
t3 = t3[["datetime_utc", "t2m"]].set_index("datetime_utc")
t6 = t6[["datetime_utc", "t2m"]].set_index("datetime_utc")

# Enforce expected grids (4 points/day) and fill real gaps
t3_reg = fix_by_previous_day_for_index(
    t3, "t2m", "T2M F3", EXPECTED_F3_INDEX
)
t6_reg = fix_by_previous_day_for_index(
    t6, "t2m", "T2M F6", EXPECTED_F6_INDEX
)

# ============================
# 2) Combine into full 3-hourly instantaneous T2M
# ============================

s3 = t3_reg["t2m"].sort_index()
s6 = t6_reg["t2m"].sort_index()

# Union of the two series: 0,3,6,...,21 every day
s_all = pd.concat([s3, s6]).sort_index()
s_all = s_all.groupby(level=0).mean()  # just in case of duplicates

df3h = s_all.to_frame(name="t2m_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["t2m_3h"].isna().sum()
print("\n=== Combined 3-hour T2M grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["t2m_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour T2M file
df3h_reset = df3h.reset_index().rename(columns={"t2m_3h": "t2m"})
df3h_reset.to_csv(OUT_T2M_3H, index=False)
print(f"\nSaved 3-hour combined T2M file to: {OUT_T2M_3H}")

# ============================
# 3) Build hourly T2M by linear interpolation
# ============================

hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)
df1h = pd.DataFrame(index=hourly_index)
df1h["t2m"] = df3h["t2m_3h"]

missing_hours_before = df1h["t2m"].isna().sum()
print("\n=== Hourly T2M before interpolation (full 2024) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing t2m values      : {missing_hours_before}")

if missing_hours_before > 0:
    missing_times = df1h.index[df1h["t2m"].isna()]
    print("First missing hourly timestamps:")
    print(missing_times[:10])

# Linear interpolation (instantaneous field) + edge ffill/bfill
df1h["t2m"] = df1h["t2m"].interpolate(method="linear")
df1h["t2m"] = df1h["t2m"].ffill().bfill()

final_missing = df1h["t2m"].isna().sum()
print(f"\nMissing values AFTER interpolation + ffill/bfill: {final_missing}")

df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_T2M_1H, index=False)
print(f"Final hourly T2M saved to: {OUT_T2M_1H}")

df1h_final.head()


## 3.2 U100 hourly reconstruction


In [ ]:
import pandas as pd
import numpy as np

# =========================================
# CONFIG – file paths for U100
# =========================================
PATH_U100_F3 = str(GFS_SOURCE / "u100/u100_f3_2024.csv")
PATH_U100_F6 = str(GFS_SOURCE / "u100/u100_f6_2024.csv")

OUT_U100_3H = str(GFS_WORK / "u100_3hour_2024_combined.csv")
OUT_U100_1H = str(GFS_WORK / "u100_hourly_2024_from_f3f6.csv")

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 valid times: 03, 09, 15, 21 (4 per day)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 valid times: 00, 06, 12, 18 (4 per day)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# =========================================
# Helper: fix missing by previous day and report
# =========================================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# =========================================
# 1) Read raw U100 f3 and f6 files
# =========================================
u3 = pd.read_csv(PATH_U100_F3)
u6 = pd.read_csv(PATH_U100_F6)

# Build datetime_utc from "day" + "valid_time"
u3["datetime_utc"] = pd.to_datetime(
    u3["day"].astype(str) + " " + u3["valid_time"].astype(str),
    utc=True
)
u6["datetime_utc"] = pd.to_datetime(
    u6["day"].astype(str) + " " + u6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
# (assumes the column with wind at 100 m is named 'u100')
u3 = u3[["datetime_utc", "u100"]].set_index("datetime_utc")
u6 = u6[["datetime_utc", "u100"]].set_index("datetime_utc")

# Enforce expected grids (4 points/day) and fill real gaps
u3_reg = fix_by_previous_day_for_index(
    u3, "u100", "U100 F3", EXPECTED_F3_INDEX
)
u6_reg = fix_by_previous_day_for_index(
    u6, "u100", "U100 F6", EXPECTED_F6_INDEX
)

# =========================================
# 2) Combine into full 3-hourly instantaneous U100
# =========================================

s3 = u3_reg["u100"].sort_index()
s6 = u6_reg["u100"].sort_index()

# Union of the two series: 0,3,6,...,21 every day
s_all = pd.concat([s3, s6]).sort_index()
s_all = s_all.groupby(level=0).mean()  # just in case of duplicates

df3h = s_all.to_frame(name="u100_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["u100_3h"].isna().sum()
print("\n=== Combined 3-hour U100 grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["u100_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour U100 file
df3h_reset = df3h.reset_index().rename(columns={"u100_3h": "u100"})
df3h_reset.to_csv(OUT_U100_3H, index=False)
print(f"\nSaved 3-hour combined U100 file to: {OUT_U100_3H}")

# =========================================
# 3) Build hourly U100 by linear interpolation
# =========================================

hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)
df1h = pd.DataFrame(index=hourly_index)
df1h["u100"] = df3h["u100_3h"]

missing_hours_before = df1h["u100"].isna().sum()
print("\n=== Hourly U100 before interpolation (full 2024) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing u100 values     : {missing_hours_before}")

if missing_hours_before > 0:
    missing_times = df1h.index[df1h["u100"].isna()]
    print("First missing hourly timestamps:")
    print(missing_times[:10])

# Linear interpolation (instantaneous field) + edge ffill/bfill
df1h["u100"] = df1h["u100"].interpolate(method="linear")
df1h["u100"] = df1h["u100"].ffill().bfill()

final_missing = df1h["u100"].isna().sum()
print(f"\nMissing values AFTER interpolation + ffill/bfill: {final_missing}")

df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_U100_1H, index=False)
print(f"Final hourly U100 saved to: {OUT_U100_1H}")

df1h_final.head()


## 3.3 V100 hourly reconstruction


In [ ]:
import pandas as pd
import numpy as np

# =========================================
# CONFIG – file paths for V100
# =========================================
PATH_V100_F3 = str(GFS_SOURCE / "v100/v100_f3_2024.csv")
PATH_V100_F6 = str(GFS_SOURCE / "v100/v100_f6_2024.csv")

OUT_V100_3H = str(GFS_WORK / "v100_3hour_2024_combined.csv")
OUT_V100_1H = str(GFS_WORK / "v100_hourly_2024_from_f3f6.csv")

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 valid times: 03, 09, 15, 21 (4 per day)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 valid times: 00, 06, 12, 18 (4 per day)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# =========================================
# Helper: fix missing by previous day and report
# =========================================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# =========================================
# 1) Read raw V100 f3 and f6 files
# =========================================
v3 = pd.read_csv(PATH_V100_F3)
v6 = pd.read_csv(PATH_V100_F6)

# Build datetime_utc from "day" + "valid_time"
v3["datetime_utc"] = pd.to_datetime(
    v3["day"].astype(str) + " " + v3["valid_time"].astype(str),
    utc=True
)
v6["datetime_utc"] = pd.to_datetime(
    v6["day"].astype(str) + " " + v6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
# (assumes the column with wind at 100 m (v-component) is named 'v100')
v3 = v3[["datetime_utc", "v100"]].set_index("datetime_utc")
v6 = v6[["datetime_utc", "v100"]].set_index("datetime_utc")

# Enforce expected grids (4 points/day) and fill real gaps
v3_reg = fix_by_previous_day_for_index(
    v3, "v100", "V100 F3", EXPECTED_F3_INDEX
)
v6_reg = fix_by_previous_day_for_index(
    v6, "v100", "V100 F6", EXPECTED_F6_INDEX
)

# =========================================
# 2) Combine into full 3-hourly instantaneous V100
# =========================================

s3 = v3_reg["v100"].sort_index()
s6 = v6_reg["v100"].sort_index()

# Union of the two series: 0,3,6,...,21 every day
s_all = pd.concat([s3, s6]).sort_index()
s_all = s_all.groupby(level=0).mean()  # just in case of duplicates

df3h = s_all.to_frame(name="v100_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["v100_3h"].isna().sum()
print("\n=== Combined 3-hour V100 grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["v100_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour V100 file
df3h_reset = df3h.reset_index().rename(columns={"v100_3h": "v100"})
df3h_reset.to_csv(OUT_V100_3H, index=False)
print(f"\nSaved 3-hour combined V100 file to: {OUT_V100_3H}")

# =========================================
# 3) Build hourly V100 by linear interpolation
# =========================================

hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)
df1h = pd.DataFrame(index=hourly_index)
df1h["v100"] = df3h["v100_3h"]

missing_hours_before = df1h["v100"].isna().sum()
print("\n=== Hourly V100 before interpolation (full 2024) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing v100 values     : {missing_hours_before}")

if missing_hours_before > 0:
    missing_times = df1h.index[df1h["v100"].isna()]
    print("First missing hourly timestamps:")
    print(missing_times[:10])

# Linear interpolation (instantaneous field) + edge ffill/bfill
df1h["v100"] = df1h["v100"].interpolate(method="linear")
df1h["v100"] = df1h["v100"].ffill().bfill()

final_missing = df1h["v100"].isna().sum()
print(f"\nMissing values AFTER interpolation + ffill/bfill: {final_missing}")

df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_V100_1H, index=False)
print(f"Final hourly V100 saved to: {OUT_V100_1H}")

df1h_final.head()


## 3.4 DSWRF hourly reconstruction

In [ ]:
import pandas as pd
import numpy as np
# Rebuild complete 2024 DSWRF from GFS f3/f6 averages by correcting gaps, deriving missing 3-hour blocks,
# and converting each 3-hour mean into consistent hourly values using block-wise disaggregation.

# ============================
# CONFIG – file paths
# ============================
PATH_F3 = str(GFS_SOURCE / "dswrf/dswrf_f3avg_2024.csv")
PATH_F6 = str(GFS_SOURCE / "dswrf/dswrf_f6avg_2024.csv")

OUT_3H = str(GFS_WORK / "dswrf_3hour_reconstructed_2024.csv")
OUT_1H = str(GFS_WORK / "dswrf_hourly_2024_from_f3f6.csv")

# Time horizon for 2024
YEAR_START       = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
YEAR_END_HOURLY  = pd.Timestamp("2024-12-31 23:00:00", tz="UTC")
YEAR_END_3H      = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")  # last 3h step

# F3 is valid at 03, 09, 15, 21 (4 per day, 6-hourly with offset 3h)
F3_START = YEAR_START + pd.Timedelta(hours=3)
F3_END   = pd.Timestamp("2024-12-31 21:00:00", tz="UTC")
EXPECTED_F3_INDEX = pd.date_range(start=F3_START, end=F3_END, freq="6h", tz="UTC")

# F6 is valid at 00, 06, 12, 18 (4 per day, 6-hourly starting at midnight)
F6_START = YEAR_START
F6_END   = pd.Timestamp("2024-12-31 18:00:00", tz="UTC")
EXPECTED_F6_INDEX = pd.date_range(start=F6_START, end=F6_END, freq="6h", tz="UTC")


# ============================
# Helper: fix missing by previous day and report
# ============================
def fix_by_previous_day_for_index(df, value_col, name, expected_index):
    df = df.sort_index()
    df_reg = df.reindex(expected_index)
    df_reg.index.name = "datetime_utc"

    missing_before = df_reg[value_col].isna().sum()
    print(f"\n=== {name}: before filling ===")
    print(f"Start (expected): {df_reg.index.min()}")
    print(f"End   (expected): {df_reg.index.max()}")
    print(f"Expected count   : {len(df_reg)}")
    print(f"Missing values   : {missing_before}")

    filled_times = []

    for ts in df_reg.index[df_reg[value_col].isna()]:
        prev = ts - pd.Timedelta(days=1)
        if prev in df_reg.index and pd.notna(df_reg.loc[prev, value_col]):
            df_reg.loc[ts, value_col] = df_reg.loc[prev, value_col]
            filled_times.append(ts)

    missing_after = df_reg[value_col].isna().sum()

    print(f"\n=== {name}: after filling from previous day ===")
    print(f"Filled from previous day: {len(filled_times)}")
    print(f"Remaining missing      : {missing_after}")

    if filled_times:
        print("First filled timestamps (copied from t-24h):")
        print(filled_times[:10])

    if missing_after > 0:
        still_missing = df_reg.index[df_reg[value_col].isna()]
        print("First still-missing timestamps:")
        print(still_missing[:10])

    return df_reg


# ============================
# 1) Read raw f3 and f6 files
# ============================
f3 = pd.read_csv(PATH_F3)
f6 = pd.read_csv(PATH_F6)

# Build datetime_utc from "day" + "valid_time"
f3["datetime_utc"] = pd.to_datetime(
    f3["day"].astype(str) + " " + f3["valid_time"].astype(str),
    utc=True
)
f6["datetime_utc"] = pd.to_datetime(
    f6["day"].astype(str) + " " + f6["valid_time"].astype(str),
    utc=True
)

# Keep only needed columns and set index
f3 = f3[["datetime_utc", "dswrf"]].rename(columns={"dswrf": "A3"}).set_index("datetime_utc")
f6 = f6[["datetime_utc", "dswrf"]].rename(columns={"dswrf": "A6"}).set_index("datetime_utc")

# Enforce the correct expected grids (4 points/day) and fill real gaps
f3_reg = fix_by_previous_day_for_index(
    f3, "A3", "F3 DSWRF", EXPECTED_F3_INDEX
)
f6_reg = fix_by_previous_day_for_index(
    f6, "A6", "F6 DSWRF", EXPECTED_F6_INDEX
)

# ============================
# 2) Reconstruct full 3-hourly series (A3_2 = 2*A6 − A3_1)
# ============================

s3 = f3_reg["A3"].sort_index()
s6 = f6_reg["A6"].sort_index()

reconstructed = {}

for t6, A6 in s6.items():
    t3_1 = t6 - pd.Timedelta(hours=3)
    if t3_1 in s3.index and pd.notna(s3.loc[t3_1]):
        A3_1 = s3.loc[t3_1]
        A3_2 = 2.0 * A6 - A3_1
        # second 3-hour block ends at t6 → assign to t6
        reconstructed[t6] = A3_2

s_rec = pd.Series(reconstructed, name="A3_rec").sort_index()

# Combine original and reconstructed A3 values
s3_full = pd.concat([s3.rename("A3_orig"), s_rec]).sort_index()
s3_full = s3_full.groupby(level=0).mean()   # handle any duplicates

df3h = s3_full.to_frame(name="dswrf_3h")
df3h.index.name = "datetime_utc"

# Force full 3-hour grid for 2024
expected_3h = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_3H,
    freq="3h",
    tz="UTC"
)
df3h = df3h.reindex(expected_3h)

missing_3h = df3h["dswrf_3h"].isna().sum()
print("\n=== Reconstructed 3-hour DSWRF grid (full 2024) ===")
print(f"Start: {df3h.index.min()}")
print(f"End  : {df3h.index.max()}")
print(f"Total 3h points : {len(df3h)}")
print(f"Missing 3h points before hourly step: {missing_3h}")

if missing_3h > 0:
    miss_times_3h = df3h.index[df3h["dswrf_3h"].isna()]
    print("First missing 3h timestamps:")
    print(miss_times_3h[:10])

# Save 3-hour reconstructed file (complete 3-hour time steps for 2024)
df3h_reset = df3h.reset_index().rename(columns={"dswrf_3h": "dswrf"})
df3h_reset.to_csv(OUT_3H, index=False)
print(f"\nSaved 3-hour reconstructed file to: {OUT_3H}")

# ============================
# ============================
# 3) Build hourly DSWRF by block-wise constant disaggregation
#    (each 3h value applies to the PREVIOUS 3 hours)
# ============================

# Full hourly index for 2024
hourly_index = pd.date_range(
    start=YEAR_START,
    end=YEAR_END_HOURLY,
    freq="1h",
    tz="UTC"
)

df1h = pd.DataFrame(index=hourly_index, columns=["dswrf"], dtype=float)

# Fill each 3-hour block: value at time t is average of [t-3, t)
for t, v in df3h["dswrf_3h"].items():
    if pd.isna(v):
        continue
    block_start = t - pd.Timedelta(hours=3)
    block_hours = pd.date_range(
        start=block_start,
        periods=3,
        freq="1h",
        tz="UTC"
    )
    for h in block_hours:
        if h in df1h.index:
            df1h.loc[h, "dswrf"] = v

# At this point, all hours from 2024-01-01 00:00 to 2024-12-31 20:00
# should be filled. Only possible NaNs are typically the very last hours
# (21:00–23:00 on Dec 31) if the last 3h block ends at 21:00.

missing_hours = df1h["dswrf"].isna().sum()
print("\n=== Hourly DSWRF after block-wise fill (before final edge fix) ===")
print(f"Total hourly timestamps : {len(df1h)}")
print(f"Missing dswrf values    : {missing_hours}")

if missing_hours > 0:
    missing_times = df1h.index[df1h["dswrf"].isna()]
    print("Missing hourly timestamps:")
    print(missing_times)

    # For the trailing night hours, it is safe to set them to 0
    # (DSWRF is zero at night).
    df1h.loc[missing_times, "dswrf"] = 0.0

# Final sanity check
final_missing = df1h["dswrf"].isna().sum()
print(f"\nMissing values AFTER block-wise fill + edge fix: {final_missing}")

# Save final hourly file
df1h_final = df1h.reset_index().rename(columns={"index": "datetime_utc"})
df1h_final.to_csv(OUT_1H, index=False)
print(f"Final hourly DSWRF (block-wise) saved to: {OUT_1H}")

df1h_final.head(10)


# 4. model-ready input preparation

This utility/precompute cell organizes the demand profiles, grid prices, GFS-based PV/wind availability, battery parameters, and emission signal files into the model-ready inputs used by M0, M1, and M2.


## 4.1 utility / precompute


In [ ]:
# Model-ready input preparation: prices, emission signals, GFS-based PV/wind,
# demand profiles and parameter metadata. Paths are set in the configuration cell.

import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------- GLOBAL CONTROLS ----------------
YEAR = 2024
N_DAYS_TO_RUN = 7
START_DAY_OF_YEAR = 1
LAMBDA_EMIS = 1.0 #ignore this

DATA_DIR = GFS_WORK
OUT_DIR = MODEL_READY_OUTPUT
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

idx_2024 = pd.date_range(
    "2024-01-01 00:00:00+00:00",
    "2024-12-31 23:00:00+00:00",
    freq="h",
    tz="UTC"
)

# ---------------- HELPERS ----------------
def read_hourly_csv(path: Path, time_col="datetime_utc") -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")
    df = pd.read_csv(path)
    if time_col not in df.columns:
        raise ValueError(f"{path.name}: missing time column '{time_col}'. Found: {list(df.columns)}")
    df[time_col] = pd.to_datetime(df[time_col], utc=True)
    df = df.sort_values(time_col).set_index(time_col)
    df = df.tz_convert("UTC") if df.index.tz is not None else df.tz_localize("UTC")
    return df

def assert_full_hourly(df: pd.DataFrame, name: str) -> pd.DataFrame:
    if df.index.duplicated().any():
        d = df.index[df.index.duplicated()][0]
        raise ValueError(f"{name}: duplicated timestamp example: {d}")
    missing = idx_2024.difference(df.index)
    extra   = df.index.difference(idx_2024)
    if len(missing) > 0:
        raise ValueError(f"{name}: missing {len(missing)} hours (first 5: {missing[:5].tolist()})")
    if len(extra) > 0:
        raise ValueError(f"{name}: extra {len(extra)} hours outside 2024 (first 5: {extra[:5].tolist()})")
    df2 = df.reindex(idx_2024)
    if df2.isna().any().any():
        cols = df2.columns[df2.isna().any()].tolist()
        where = df2[df2[cols].isna().any(axis=1)].index[:5].tolist()
        raise ValueError(f"{name}: NaNs after reindex in cols {cols}; example {where}")
    return df2

def save_lineplot_png(df: pd.DataFrame, cols, title: str, ylabel: str, outpath: Path):
    plt.figure(figsize=(14,4))
    for c in cols:
        plt.plot(df.index, df[c], label=c)
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

# =============================================================================
# ---------------- SCALARS
# =============================================================================

# ---- Site / PV geometry ----
phi_deg   = 40.25      # latitude [deg]
lam_deg   = -3.75      # longitude [deg] (east positive, west negative)
beta_deg  = 34  # tilt [deg]
gamma_deg = 0  # kept for completeness

rho_g     = 0.2  # ground albedo [-]
eta_inv   = 0.96   # inverter efficiency [-]
G_sc      = 1366   # solar constant [W/m^2]

# ---- PV efficiency model coefficients ----
p_pv = 23.62
q_pv = -0.2983
r_pv = -0.09307
s_pv = -0.9795
m_pv = 0.1912
u_pv = 0.9865
h_pv = 0.028       # cell temp coefficient [°C/(W/m^2)]
ETA_PV_MAX = 0.2
# ---- PV sizing ----
installed_pv_kwp = 120000   # [kWp]
A_PV_unit        = 7   # [m^2/kWp]

# ---- Wind model scalars ----
Z_ref   = 100    # reference height for wind speed [m]
Z_wt    = 117     # hub height [m]
alpha   = 0.207     # shear exponent [-]
rho_air = 1.225    # air density [kg/m^3]
P_r_W   = 3450000     # rated power per turbine [W]
A_WT    = 12469     # rotor swept area [m^2]
Cp_eff  = 0.29    # effective Cp [-]
eta_wf  = 0.95    # wind farm efficiency [-]
wf_avail= 0.95    # availability [-]

v_in    = 3      # cut-in [m/s]
v_rated = 11.5     # rated speed [m/s]
v_out   = 22.5    # cut-out [m/s]
N_WT    = 35   # number of turbines [int]

# ---- Plant capacities  ----
P_ELZ_MAX_MW        = 209 #mw
P_GRID_MAX_MW       = 295.58 #mw
H2_MAX_KGPH         = 3657.567
NH3_CAP_KGPH        = 13472.036
MEOH_CAP_KGPH       = 5818.856
NH3_CRACK_CAP_KGPH  = 6062.416

# ---- Demand profile parameters  ----
A_DAY_NH3  = 0.06
A_DAY_H2   = 0.1
A_DAY_MEOH = 0.05
WEEKEND_FACTOR   = 0.90
DEMAND_CLIP_FRAC = 0.90
# =============================================================================
# ---------------- LOAD INPUT SERIES ----------------
# =============================================================================
price = assert_full_hourly(
    read_hourly_csv(PRICE_SOURCE)[["da_elec_price_EURperMWh"]],
    "price"
)

aef = assert_full_hourly(
    read_hourly_csv(SIGNAL_SOURCE / "grid_aef_hourly_2024.csv")[["grid_aef_predicted_kg_per_kwh"]],
    "AEF"
)

AEF_COL = "grid_aef_predicted_kg_per_kwh"
MEF_UP_COL = "grid_mef_upramp_pred_kg_per_kwh"
MEF_SOFT_COL = "grid_mef_soft_kg_per_kwh"

# ---- Tier 2.2 MEF (upramp) fill with AEF ----
mef_upramp_raw = read_hourly_csv(
    SIGNAL_SOURCE / "grid_mef_upramp_hourly_2024.csv"
)[[MEF_UP_COL]]
mef_upramp = mef_upramp_raw.reindex(idx_2024)

missing_mask_up = mef_upramp[MEF_UP_COL].isna()
missing_hours_up = mef_upramp.index[missing_mask_up]
print(f"[MEF-UP FILL] Missing MEF-upramp hours: {int(missing_mask_up.sum())} / {len(mef_upramp)}")
if missing_mask_up.sum() > 0:
    print("[MEF-UP FILL] First 10 missing timestamps:", missing_hours_up[:10].tolist())

mef_upramp_fill_audit = pd.DataFrame({
    "datetime_utc": missing_hours_up,
    "aef_used": aef.loc[missing_mask_up, AEF_COL].values,
    "fill_reason": "MEF upramp missing/undefined -> filled with AEF"
})
mef_upramp.loc[missing_mask_up, MEF_UP_COL] = aef.loc[missing_mask_up, AEF_COL].values

mef_upramp_fill_audit.to_csv(OUT_DIR / "mef_upramp_filled_with_aef_hours_2024.csv", index=False)
print(f"[MEF-UP FILL] Saved audit: {OUT_DIR / 'mef_upramp_filled_with_aef_hours_2024.csv'}")

if mef_upramp.isna().any().any():
    where = mef_upramp[mef_upramp.isna().any(axis=1)].index[:5].tolist()
    raise ValueError(f"MEF_UPRAMP: still has NaNs after fill. Example times: {where}")

# ---- Tier 2.3 MEF (soft) fill with AEF ----
mef_soft_raw = read_hourly_csv(
    SIGNAL_SOURCE / "grid_mef_soft_hourly_2024.csv"
)[[MEF_SOFT_COL]]
mef_soft = mef_soft_raw.reindex(idx_2024)

missing_mask_soft = mef_soft[MEF_SOFT_COL].isna()
missing_hours_soft = mef_soft.index[missing_mask_soft]
print(f"[MEF-SOFT FILL] Missing MEF-soft hours: {int(missing_mask_soft.sum())} / {len(mef_soft)}")
if missing_mask_soft.sum() > 0:
    print("[MEF-SOFT FILL] First 10 missing timestamps:", missing_hours_soft[:10].tolist())

mef_soft_fill_audit = pd.DataFrame({
    "datetime_utc": missing_hours_soft,
    "aef_used": aef.loc[missing_mask_soft, AEF_COL].values,
    "fill_reason": "MEF soft missing/undefined -> filled with AEF"
})
mef_soft.loc[missing_mask_soft, MEF_SOFT_COL] = aef.loc[missing_mask_soft, AEF_COL].values

mef_soft_fill_audit.to_csv(OUT_DIR / "mef_soft_filled_with_aef_hours_2024.csv", index=False)
print(f"[MEF-SOFT FILL] Saved audit: {OUT_DIR / 'mef_soft_filled_with_aef_hours_2024.csv'}")

if mef_soft.isna().any().any():
    where = mef_soft[mef_soft.isna().any(axis=1)].index[:5].tolist()
    raise ValueError(f"MEF_SOFT: still has NaNs after fill. Example times: {where}")


dswrf = assert_full_hourly(
    read_hourly_csv(GFS_WORK / "dswrf_hourly_2024_from_f3f6.csv")[["dswrf"]],
    "DSWRF"
)
t2m = assert_full_hourly(
    read_hourly_csv(GFS_WORK / "t2m_hourly_2024_from_f3f6.csv")[["t2m"]],
    "T2M"
)
u100 = assert_full_hourly(
    read_hourly_csv(GFS_WORK / "u100_hourly_2024_from_f3f6.csv")[["u100"]],
    "U100"
)
v100 = assert_full_hourly(
    read_hourly_csv(GFS_WORK / "v100_hourly_2024_from_f3f6.csv")[["v100"]],
    "V100"
)

print("All input CSVs validated.")

# =============================================================================
# ---------------- PV MODEL (with EoT; efficiency clip [0,1]) -------
# =============================================================================

# sizing
A_PV             = float(A_PV_unit) * float(installed_pv_kwp)
PV_NAMEPLATE_MWp = float(installed_pv_kwp) / 1000.0

# inputs
G_t   = np.clip(dswrf["dswrf"].astype(float).values, 0.0, None)
T_a_C = (t2m["t2m"].astype(float).values - 273.15)

# radians
phi  = np.deg2rad(float(phi_deg))
beta = np.deg2rad(float(beta_deg))

# day of year
n = idx_2024.dayofyear.values.astype(float)

# declination
delta = np.deg2rad(23.45 * np.sin(np.deg2rad(360.0*(284.0 + n)/365.0)))

# ---- Equation of Time ----
# B = (360/365)*(d - 81)   [deg]
# EoT = 9.87 sin(2B) - 7.53 cos(B) - 1.5 sin(B)   [minutes]
B_deg = (360.0/365.0) * (n - 81.0)
B_rad = np.deg2rad(B_deg)
EoT_min = 9.87*np.sin(2.0*B_rad) - 7.53*np.cos(B_rad) - 1.5*np.sin(B_rad)

# solar time ts: UTC hour + longitude correction + EoT correction
ts = idx_2024.hour.values.astype(float) + (float(lam_deg)/15.0) + (EoT_min/60.0)

# hour angle
omega = np.deg2rad(15.0*(ts - 12.0))

# zenith
cos_theta_z = np.sin(phi)*np.sin(delta) + np.cos(phi)*np.cos(delta)*np.cos(omega)
cos_theta_z = np.clip(cos_theta_z, 0.0, 1.0)
theta_z = np.arccos(cos_theta_z)

# incidence on tilted plane
cos_theta_t = np.sin(delta)*np.sin(phi - beta) + np.cos(delta)*np.cos(phi - beta)*np.cos(omega)
cos_theta_t = np.clip(cos_theta_t, 0.0, 1.0)

# extraterrestrial horizontal proxy
H0_t = float(G_sc)*(1.0 + 0.033*np.cos(np.deg2rad(360.0*n/365.0))) * (
    np.cos(phi)*np.cos(delta)*np.cos(omega) + np.sin(phi)*np.sin(delta)
)
H0_t = np.clip(H0_t, 1e-6, None)

# clearness index
CI = np.clip(G_t / H0_t, 0.0, 2.0)

# diffuse fraction f(CI)
f = np.zeros_like(CI)
m1 = CI < 0.21
m2 = (CI >= 0.21) & (CI <= 0.76)
m3 = CI > 0.76
f[m1] = 0.995 - 0.081*CI[m1]
f[m2] = 0.724 + 2.738*CI[m2] - 8.32*(CI[m2]**2) + 4.967*(CI[m2]**3)
f[m3] = 0.18
f = np.clip(f, 0.0, 1.0)

G_d  = f * G_t
G_bh = np.clip(G_t - G_d, 0.0, None)
F_t = 1.0 - (f**2)

# diffuse on tilt
G_db = (
    G_d
    * 0.5 * (1.0 + np.cos(beta))
    * (1.0 + F_t * (np.sin(beta / 2.0) ** 3))
    * (1.0 + F_t * (cos_theta_t ** 2) * (np.sin(theta_z) ** 3))
)

# beam on tilt
G_bb = np.zeros_like(G_t)
beam_mask = cos_theta_z >= 0.05
G_bb[beam_mask] = G_bh[beam_mask] * (cos_theta_t[beam_mask] / cos_theta_z[beam_mask])
G_bb = np.clip(G_bb, 0.0, None)

# ground reflected
G_rp = float(rho_g) * G_t * (1.0 - np.cos(beta))/2.0
G_beta = np.clip(G_db + G_bb + G_rp, 0.0, None)

# air mass
AM = 1.0 / (np.cos(theta_z) + 0.50572*((96.07995 - np.rad2deg(theta_z))**(-1.6364)))
#AM = np.clip(AM, 1.0,2.0)

# cell temp
theta_cell = T_a_C + float(h_pv) * G_beta
G0, theta0, AM0 = 1000.0, 25.0, 1.5
xG = (G_beta / G0)
xAM = (AM / AM0)
# efficiency model (percent to fraction)
eta_pv_percent = float(p_pv) * (
    (float(q_pv) * xG + (xG ** float(m_pv))) *
    (1.0 + float(r_pv) * (theta_cell / theta0) + float(s_pv) * xAM + (xAM ** float(u_pv)))
)

# convert percent -> fraction and clip(clip is redundant)
eta_pv = np.clip(eta_pv_percent / 100.0, 0.0, ETA_PV_MAX)

# ---- power (paper Eq. 14) ----
P_pv_dc_W = G_beta * eta_pv * A_PV
P_pv_ac_W = P_pv_dc_W * float(eta_inv)

pv_mw_raw = np.maximum(P_pv_ac_W / 1e6, 0.0)

# NOTE: clipping to nameplate is an "available capacity cap" assumption.
# If you want "physics-only" output, do NOT cap here; instead cap only when aggregating system export.
pv_mw = np.minimum(pv_mw_raw, PV_NAMEPLATE_MWp)

# ---------------- WIND MODEL (piecewise: cubic to v_rated, then constant P_r to v_out) ----------------

# 1) wind speed magnitude at reference height
v_ref = np.sqrt(u100["u100"].values**2 + v100["v100"].values**2)

# 2) extrapolate to hub height using power law
v_hub = v_ref * (float(Z_wt) / float(Z_ref))**float(alpha)

# 3) aerodynamic cubic power (proxy) up to rated
P_aero = 0.5 * float(rho_air) * float(A_WT) * float(Cp_eff) * (v_hub**3)

# 4) piecewise turbine electrical power (before efficiencies)
P_raw = np.zeros_like(v_hub, dtype=float)

# Region A: cut-in to rated -> cubic
mask_cubic = (v_hub >= float(v_in)) & (v_hub < float(v_rated))
P_raw[mask_cubic] = P_aero[mask_cubic]

# Region B: rated to cut-out -> constant rated power
mask_rated = (v_hub >= float(v_rated)) & (v_hub < float(v_out))
P_raw[mask_rated] = float(P_r_W)

# Region C: below cut-in or above/equal cut-out -> already 0 by initialization

# 5) apply wind farm efficiency and availability
P_one = P_raw * float(eta_wf) * float(wf_avail)

# 6) aggregate turbines and cap at farm nameplate
wind_mw_raw = np.clip((float(N_WT) * P_one) / 1e6, 0.0, None)
WIND_NAMEPLATE_MW = float(N_WT) * (float(P_r_W) / 1e6)
wind_mw = np.minimum(wind_mw_raw, WIND_NAMEPLATE_MW)


# ---------------- SAVE RES TIMESERIES ----------------
res = pd.DataFrame(
    {"pv_mw": pv_mw, "wind_mw": wind_mw, "res_total_mw": pv_mw + wind_mw},
    index=idx_2024
)
res.to_csv(OUT_DIR / "res_hourly_2024.csv", index_label="datetime_utc")

# =============================================================================
# ---------------- CAPACITIES
# =============================================================================
pd.DataFrame([
    ["P_ELZ_MAX_MW", float(P_ELZ_MAX_MW), "MW"],
    ["P_GRID_MAX_MW", float(P_GRID_MAX_MW), "MW"],
    ["H2_MAX_KGPH", float(H2_MAX_KGPH), "kg/h"],
    ["NH3_CAP_KGPH", float(NH3_CAP_KGPH), "kg/h"],
    ["MEOH_CAP_KGPH", float(MEOH_CAP_KGPH), "kg/h"],
    ["NH3_CRACK_CAP_KGPH", float(NH3_CRACK_CAP_KGPH), "kg/h"],
], columns=["Symbol","Value","Unit"]).to_csv(OUT_DIR / "derived_capacities.csv", index=False)


# =============================================================================
# ---------------- DEMAND PROFILES (EUROSTAT BACKBONE + SMOOTH RAMPS) ---------
# =============================================================================

rng = np.random.default_rng(42)

def ar1(n, phi=0.97, sigma=0.02, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    e = rng.normal(0.0, sigma, size=n)
    x = np.zeros(n)
    for i in range(1, n):
        x[i] = phi * x[i-1] + e[i]
    return x

def spikes(n, p=0.002, mag=0.08, rng=None):
    if rng is None:
        rng = np.random.default_rng(0)
    u = rng.random(n)
    s = np.zeros(n)
    hit = u < p
    s[hit] = rng.uniform(0.0, mag, size=hit.sum())
    return s

def enforce_min_mean_max(x, target_min, target_mean, target_max, max_iter=60):
    """
    Enforce exact min/mean/max (within float tolerance) by:
    1) mapping x -> z in [0,1] using normalization
    2) applying z^p (monotone) to tune the mean without changing min/max
    3) scaling to [target_min, target_max]
    """
    x = np.asarray(x, dtype=float)

    xmin = float(np.min(x))
    xmax = float(np.max(x))
    if xmax - xmin < 1e-12:
        z = np.full_like(x, 0.5)
    else:
        z = (x - xmin) / (xmax - xmin)
    z = np.clip(z, 0.0, 1.0)

    lo, hi = 0.05, 20.0
    for _ in range(max_iter):
        mid = 0.5 * (lo + hi)
        y_mid = target_min + (target_max - target_min) * (z ** mid)
        m_mid = float(np.mean(y_mid))
        if m_mid > target_mean:
            lo = mid
        else:
            hi = mid

    p_star = 0.5 * (lo + hi)
    y = target_min + (target_max - target_min) * (z ** p_star)

    i_min = int(np.argmin(y))
    i_max = int(np.argmax(y))
    y[i_min] = target_min
    y[i_max] = target_max

    mean_err = target_mean - float(np.mean(y))
    y = np.clip(y + mean_err, target_min, target_max)
    y[i_min] = target_min
    y[i_max] = target_max

    return y

def block_mean_repeat(x, block=3):
    """
    Enforce 'ramps' by removing hour-to-hour jitter:
    - average each block of 'block' hours
    - repeat that average across the block
    2024 has 8784 hours; 8784 % 3 == 0 so block=3 is safe.
    """
    x = np.asarray(x, dtype=float)
    n = len(x)
    if n % block != 0:
        # fallback: trim last remainder (should not happen for 2024 with block=3)
        n2 = n - (n % block)
        x = x[:n2]
        n = n2
    xb = x.reshape(-1, block).mean(axis=1)
    return np.repeat(xb, block)

# ---------------- EUROSTAT NORMALIZED BACKBONES (hard-coded) ----------------
# Extracted from:
#  - csv_eurostat-passengers port.csv (quarterly, Spain, 2024)
#  - csv_eurostat-gas consumbtion inland.csv (monthly, Spain, 2024)
# Normalization: mean = 1.0, then damped towards 1.0 to avoid extreme swings.

# Q1..Q4 multipliers for NH3 (port passengers), damped
NH3_PORT_Q_MULT_2024 = np.array([0.822047, 0.896977, 1.340477, 0.940499], dtype=float)

# Jan..Dec multipliers for H2 (inland gas consumption), damped
H2_GAS_M_MULT_2024 = np.array([
    0.985216, 0.984325, 1.025127, 1.014627, 0.948715, 0.952309,
    0.952821, 0.958588, 0.993616, 1.021583, 1.076420, 1.086653
], dtype=float)

# Hourly backbones
months = idx_2024.month.values.astype(int)
quarters = ((months - 1) // 3).astype(int)  # 0..3
nh3_backbone = NH3_PORT_Q_MULT_2024[quarters]
nh3_backbone = pd.Series(nh3_backbone, index=idx_2024).rolling(24*30, min_periods=1, center=True).mean().values

h2_backbone  = H2_GAS_M_MULT_2024[months - 1]
meoh_backbone = np.ones(len(idx_2024), dtype=float)  # explicitly flat (per your instruction)

# ---------------- base realism drivers  -------
wfac = np.where(idx_2024.weekday.values >= 5, WEEKEND_FACTOR, 1.0)

Ta_C = pd.Series((t2m["t2m"].astype(float).values - 273.15), index=idx_2024)
Ta_C_smooth = Ta_C.rolling(24*7, center=True, min_periods=1).mean()
T_ref = float(Ta_C_smooth.quantile(0.75))
H = np.clip((T_ref - Ta_C_smooth) / 15.0, 0.0, 1.0)

uplift_max = 0.12
temp_uplift = (1.0 + uplift_max * H.values)

t = np.arange(len(idx_2024))
daily = np.sin(2*np.pi*t/24.0)

shape_nh3  = nh3_backbone * wfac * temp_uplift * (1.0 + float(A_DAY_NH3)*daily)
shape_h2   = h2_backbone  * wfac * temp_uplift * (1.0 + float(A_DAY_H2)*daily)
shape_meoh = meoh_backbone * wfac * temp_uplift * (1.0 + float(A_DAY_MEOH)*daily)

# ---------------- low-frequency stochasticity + smoothing to enforce ramps ----
# Noise is added, then block-smoothed (3h) to avoid hour-to-hour “teleporting”.
eps_nh3  = ar1(len(idx_2024), phi=0.990, sigma=0.008, rng=rng) + spikes(len(idx_2024), p=0.0015, mag=0.05, rng=rng)
eps_h2   = ar1(len(idx_2024), phi=0.985, sigma=0.010, rng=rng) + spikes(len(idx_2024), p=0.0015, mag=0.06, rng=rng)
eps_meoh = ar1(len(idx_2024), phi=0.985, sigma=0.010, rng=rng) + spikes(len(idx_2024), p=0.0015, mag=0.06, rng=rng)

nh3_raw = shape_nh3  * (1.0 + eps_nh3)
h2_raw  = shape_h2   * (1.0 + eps_h2)
me_raw  = shape_meoh * (1.0 + eps_meoh)

# Enforce “ramps” by 3-hour blocking + light rolling smoothing
BLOCK_H = 3

nh3_s = pd.Series(block_mean_repeat(nh3_raw, BLOCK_H), index=idx_2024).rolling(BLOCK_H, min_periods=1).mean().values
h2_s  = pd.Series(block_mean_repeat(h2_raw,  BLOCK_H), index=idx_2024).rolling(BLOCK_H, min_periods=1).mean().values
me_s  = pd.Series(block_mean_repeat(me_raw,  BLOCK_H), index=idx_2024).rolling(BLOCK_H, min_periods=1).mean().values

# ---------------- FINAL TARGETS (kg/h) --------------------------------------
H2_MIN,   H2_MEAN,   H2_MAX   = 700,  1400, 2000
NH3_MIN,  NH3_MEAN,  NH3_MAX  = 800,  1500, 3000
MEOH_MIN, MEOH_MEAN, MEOH_MAX = 600,  1200, 1800

h2  = enforce_min_mean_max(h2_s,  H2_MIN,  H2_MEAN,  H2_MAX)
nh3 = enforce_min_mean_max(nh3_s, NH3_MIN, NH3_MEAN, NH3_MAX)
me  = enforce_min_mean_max(me_s,  MEOH_MIN, MEOH_MEAN, MEOH_MAX)

dem = pd.DataFrame({
    "nh3_demand_kgph": nh3,
    "h2_demand_kgph":  h2,
    "meoh_demand_kgph": me,
}, index=idx_2024)

dem.to_csv(OUT_DIR / "demand_hourly_2024.csv", index_label="datetime_utc")


def _chk(col, mn, me, mx):
    v = dem[col].values.astype(float)
    if not (abs(v.min()-mn) < 1e-6 and abs(v.mean()-me) < 1e-3 and abs(v.max()-mx) < 1e-6):
        raise ValueError(f"{col}: got min/mean/max = {v.min():.6f}/{v.mean():.6f}/{v.max():.6f}, expected {mn}/{me}/{mx}")

_chk("h2_demand_kgph",   H2_MIN,   H2_MEAN,   H2_MAX)
_chk("nh3_demand_kgph",  NH3_MIN,  NH3_MEAN,  NH3_MAX)
_chk("meoh_demand_kgph", MEOH_MIN, MEOH_MEAN, MEOH_MAX)



# ---------------- SAVE SERIES ----------------
price.to_csv(OUT_DIR / "grid_price_hourly_2024.csv", index_label="datetime_utc")
aef.to_csv(OUT_DIR / "grid_aef_hourly_2024.csv", index_label="datetime_utc")
mef_upramp.to_csv(OUT_DIR / "grid_mef_upramp_hourly_2024.csv", index_label="datetime_utc")
mef_soft.to_csv(OUT_DIR / "grid_mef_soft_hourly_2024.csv", index_label="datetime_utc")

# ---------------- PLOTS (PNG) ----------------
# ---------------- PLOTS (PNG) ----------------

def save_lineplot_png_resampled(df: pd.DataFrame, col: str, rule: str, title: str, ylabel: str, outpath: Path):
    s = df[col].resample(rule).mean()
    plt.figure(figsize=(14,4))
    plt.plot(s.index, s.values, label=f"{col} ({rule} mean)")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

def save_multi_lineplot_png_resampled(df: pd.DataFrame, cols, rule: str, title: str, ylabel: str, outpath: Path):
    df_r = df[cols].resample(rule).mean()
    plt.figure(figsize=(14,4))
    for c in cols:
        plt.plot(df_r.index, df_r[c].values, label=f"{c} ({rule} mean)")
    plt.title(title)
    plt.ylabel(ylabel)
    plt.legend()
    plt.tight_layout()
    plt.savefig(outpath, dpi=150)
    plt.close()

# ---- base series (hourly) ----
# res: pv_mw, wind_mw, res_total_mw  (already hourly)
# dem: h2_demand_kgph, nh3_demand_kgph, meoh_demand_kgph (already hourly)

# ---- PV diagnostic series (hourly) ----
pv_diag = pd.DataFrame({
    "dswrf_Wm2": dswrf["dswrf"].astype(float),
    "G_beta_Wm2": pd.Series(G_beta, index=idx_2024),
    "eta_pv_frac": pd.Series(eta_pv, index=idx_2024),
    "theta_cell_C": pd.Series(theta_cell, index=idx_2024),
    "AM": pd.Series(AM, index=idx_2024),
}, index=idx_2024)

# =========================
# 1) RES plots (hourly)
# =========================
save_lineplot_png(res, ["pv_mw"], "Available PV generation (full year, hourly)", "MW", PLOT_DIR / "pv_hourly_full_year.png")
save_lineplot_png(res, ["wind_mw"], "Available wind generation (full year, hourly)", "MW", PLOT_DIR / "wind_hourly_full_year.png")
save_lineplot_png(res, ["res_total_mw"], "Available total RES generation (full year, hourly)", "MW", PLOT_DIR / "res_total_hourly_full_year.png")


save_lineplot_png(res, ["pv_mw", "wind_mw", "res_total_mw"], "Available RES generation (full year, hourly)", "MW", PLOT_DIR / "res_all_hourly_full_year.png")

# 2-week zoom (hourly)
zoom = slice(res.index[0], res.index[0] + pd.Timedelta(days=14))
save_lineplot_png(res.loc[zoom], ["pv_mw"], "Available PV generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "pv_hourly_first_2_weeks.png")
save_lineplot_png(res.loc[zoom], ["wind_mw"], "Available wind generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "wind_hourly_first_2_weeks.png")
save_lineplot_png(res.loc[zoom], ["res_total_mw"], "Available total RES generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "res_total_hourly_first_2_weeks.png")
save_lineplot_png(res.loc[zoom], ["pv_mw", "wind_mw", "res_total_mw"], "Available RES generation (first 2 weeks, hourly)", "MW", PLOT_DIR / "res_all_hourly_first_2_weeks.png")

# =========================
# 2) RES plots (daily mean)
# =========================
for c, fname, title in [
    ("pv_mw", "pv_daily_mean.png", "PV generation (daily mean)"),
    ("wind_mw", "wind_daily_mean.png", "Wind generation (daily mean)"),
    ("res_total_mw", "res_total_daily_mean.png", "Total RES generation (daily mean)"),
]:
    save_lineplot_png_resampled(res, c, "D", title, "MW", PLOT_DIR / fname)


save_multi_lineplot_png_resampled(
    res, ["pv_mw", "wind_mw", "res_total_mw"], "D",
    "RES generation (daily mean)", "MW",
    PLOT_DIR / "res_all_daily_mean.png"
)

# =========================
# 3) RES plots (monthly mean)
# =========================
for c, fname, title in [
    ("pv_mw", "pv_monthly_mean.png", "PV generation (monthly mean)"),
    ("wind_mw", "wind_monthly_mean.png", "Wind generation (monthly mean)"),
    ("res_total_mw", "res_total_monthly_mean.png", "Total RES generation (monthly mean)"),
]:
    save_lineplot_png_resampled(res, c, "MS", title, "MW", PLOT_DIR / fname)


save_multi_lineplot_png_resampled(
    res, ["pv_mw", "wind_mw", "res_total_mw"], "MS",
    "RES generation (monthly mean)", "MW",
    PLOT_DIR / "res_all_monthly_mean.png"
)

# =========================
# 4) Demand plots (hourly full-year) — separate per product + combined
# =========================
save_lineplot_png(dem, ["h2_demand_kgph"],  "H2 demand (full year, hourly)",  "kg/h", PLOT_DIR / "demand_h2_hourly_full_year.png")
save_lineplot_png(dem, ["nh3_demand_kgph"], "NH3 demand (full year, hourly)", "kg/h", PLOT_DIR / "demand_nh3_hourly_full_year.png")
save_lineplot_png(dem, ["meoh_demand_kgph"],"MeOH demand (full year, hourly)","kg/h", PLOT_DIR / "demand_meoh_hourly_full_year.png")


save_lineplot_png(
    dem, ["h2_demand_kgph", "nh3_demand_kgph", "meoh_demand_kgph"],
    "Demand profiles (full year, hourly)", "kg/h",
    PLOT_DIR / "demand_all_hourly_full_year.png"
)

# =========================
# 5) Demand plots (daily mean)
# =========================
for c, fname, title in [
    ("h2_demand_kgph",  "demand_h2_daily_mean.png",  "H2 demand (daily mean)"),
    ("nh3_demand_kgph", "demand_nh3_daily_mean.png", "NH3 demand (daily mean)"),
    ("meoh_demand_kgph","demand_meoh_daily_mean.png","MeOH demand (daily mean)"),
]:
    save_lineplot_png_resampled(dem, c, "D", title, "kg/h", PLOT_DIR / fname)

save_multi_lineplot_png_resampled(
    dem, ["h2_demand_kgph", "nh3_demand_kgph", "meoh_demand_kgph"], "D",
    "Demand profiles (daily mean)", "kg/h",
    PLOT_DIR / "demand_all_daily_mean.png"
)

# =========================
# 6) Demand plots (monthly mean)
# =========================
for c, fname, title in [
    ("h2_demand_kgph",  "demand_h2_monthly_mean.png",  "H2 demand (monthly mean)"),
    ("nh3_demand_kgph", "demand_nh3_monthly_mean.png", "NH3 demand (monthly mean)"),
    ("meoh_demand_kgph","demand_meoh_monthly_mean.png","MeOH demand (monthly mean)"),
]:
    save_lineplot_png_resampled(dem, c, "MS", title, "kg/h", PLOT_DIR / fname)

save_multi_lineplot_png_resampled(
    dem, ["h2_demand_kgph", "nh3_demand_kgph", "meoh_demand_kgph"], "MS",
    "Demand profiles (monthly mean)", "kg/h",
    PLOT_DIR / "demand_all_monthly_mean.png"
)

# =========================
# 7) PV model sanity plots (hourly / daily / monthly)
# =========================
# Hourly full-year
for c, fname, ylabel, title in [
    ("dswrf_Wm2",     "pv_diag_dswrf_hourly_full_year.png", "W/m²", "DSWRF input (full year, hourly)"),
    ("G_beta_Wm2",    "pv_diag_Gbeta_hourly_full_year.png", "W/m²", "Plane-of-array irradiance G_beta (full year, hourly)"),
    ("eta_pv_frac",   "pv_diag_eta_hourly_full_year.png",   "-",    "PV efficiency eta_pv (fraction, full year, hourly)"),
    ("theta_cell_C",  "pv_diag_tcell_hourly_full_year.png", "°C",   "Cell temperature theta_cell (full year, hourly)"),
    ("AM",            "pv_diag_AM_hourly_full_year.png",    "-",    "Air mass AM (full year, hourly)"),
]:
    save_lineplot_png(pv_diag, [c], title, ylabel, PLOT_DIR / fname)

# Daily mean
for c, fname, ylabel, title in [
    ("dswrf_Wm2",     "pv_diag_dswrf_daily_mean.png", "W/m²", "DSWRF input (daily mean)"),
    ("G_beta_Wm2",    "pv_diag_Gbeta_daily_mean.png", "W/m²", "G_beta (daily mean)"),
    ("eta_pv_frac",   "pv_diag_eta_daily_mean.png",   "-",    "eta_pv (daily mean)"),
    ("theta_cell_C",  "pv_diag_tcell_daily_mean.png", "°C",   "theta_cell (daily mean)"),
    ("AM",            "pv_diag_AM_daily_mean.png",    "-",    "AM (daily mean)"),
]:
    save_lineplot_png_resampled(pv_diag, c, "D", title, ylabel, PLOT_DIR / fname)

# Monthly mean
for c, fname, ylabel, title in [
    ("dswrf_Wm2",     "pv_diag_dswrf_monthly_mean.png", "W/m²", "DSWRF input (monthly mean)"),
    ("G_beta_Wm2",    "pv_diag_Gbeta_monthly_mean.png", "W/m²", "G_beta (monthly mean)"),
    ("eta_pv_frac",   "pv_diag_eta_monthly_mean.png",   "-",    "eta_pv (monthly mean)"),
    ("theta_cell_C",  "pv_diag_tcell_monthly_mean.png", "°C",   "theta_cell (monthly mean)"),
    ("AM",            "pv_diag_AM_monthly_mean.png",    "-",    "AM (monthly mean)"),
]:
    save_lineplot_png_resampled(pv_diag, c, "MS", title, ylabel, PLOT_DIR / fname)


# ---------------- PARAMS JSON (for scenario cells) ----------------
# Note: removed SCALARS_FROM_UTILITY_INFO_FILE because Excel read is removed.
params = {
    "YEAR": YEAR,
    "N_DAYS_TO_RUN": N_DAYS_TO_RUN,
    "START_DAY_OF_YEAR": START_DAY_OF_YEAR,
    "LAMBDA_EMIS": LAMBDA_EMIS,
    "KEY_CAPS": {
        "P_ELZ_MAX_MW": float(P_ELZ_MAX_MW),
        "P_GRID_MAX_MW": float(P_GRID_MAX_MW),
        "H2_MAX_KGPH": float(H2_MAX_KGPH),
        "NH3_CAP_KGPH": float(NH3_CAP_KGPH),
        "MEOH_CAP_KGPH": float(MEOH_CAP_KGPH),
        "NH3_CRACK_CAP_KGPH": float(NH3_CRACK_CAP_KGPH),
    },
    "PV": {
        "A_PV_m2": float(A_PV),
        "PV_NAMEPLATE_MWp": float(PV_NAMEPLATE_MWp),
    },
    "WIND": {
        "N_WT": int(float(N_WT)),
        "WIND_NAMEPLATE_MW": float(WIND_NAMEPLATE_MW),
    },
    "FILES_WRITTEN": {
            "res_hourly_2024": str(OUT_DIR / "res_hourly_2024.csv"),
            "demand_hourly_2024": str(OUT_DIR / "demand_hourly_2024.csv"),
            "grid_price_hourly_2024": str(OUT_DIR / "grid_price_hourly_2024.csv"),
            "grid_aef_hourly_2024": str(OUT_DIR / "grid_aef_hourly_2024.csv"),
            "grid_mef_upramp_hourly_2024": str(OUT_DIR / "grid_mef_upramp_hourly_2024.csv"),
            "grid_mef_soft_hourly_2024": str(OUT_DIR / "grid_mef_soft_hourly_2024.csv"),
            "mef_upramp_fill_audit": str(OUT_DIR / "mef_upramp_filled_with_aef_hours_2024.csv"),
            "mef_soft_fill_audit": str(OUT_DIR / "mef_soft_filled_with_aef_hours_2024.csv"),
            "plots_dir": str(PLOT_DIR),
    }
}

with open(OUT_DIR / "params_scenario_all.json", "w") as f:
    json.dump(params, f, indent=2)

print(f"[JSON] Saved: {OUT_DIR / 'params_scenario_all.json'}")

# ---------------- SANITY PRINTS ----------------
print("\n========== Utility V3 summary ==========")
print("Saved to:", OUT_DIR)
print(" - res_hourly_2024.csv")
print(" - demand_hourly_2024.csv")
print(" - grid_price_hourly_2024.csv")
print(" - grid_aef_hourly_2024.csv")
print(" - grid_mef_upramp_hourly_2024.csv")
print(" - grid_mef_soft_hourly_2024.csv")
print(" - derived_capacities.csv")
print(" - params_scenario_all.json")
print(" - plots/*.png")

print("\nRES stats (MW):")
print(res[["pv_mw","wind_mw","res_total_mw"]].describe().loc[["min","mean","max"]])

tmp = pd.DataFrame({"G_t": G_t, "G_beta": G_beta, "eta_pv": eta_pv}, index=idx_2024)
print("\nPV intermediate sanity:")
print(tmp.describe().loc[["min","mean","max"]])

print("\nDemand stats (kg/h):")
print(dem.describe().loc[["min","mean","max"]])
print("\nScale check:")
print("PV nameplate (MWp):", PV_NAMEPLATE_MWp, " PV max after cap (MW):", float(np.max(pv_mw)))
print("Wind nameplate (MW):", WIND_NAMEPLATE_MW, " Wind max after cap (MW):", float(np.max(wind_mw)))
print("Electrolyzer max (MW):", float(P_ELZ_MAX_MW))
print("Grid import cap (MW):", float(P_GRID_MAX_MW))
clip_frac = np.mean(pv_mw_raw >= PV_NAMEPLATE_MWp)
print("Fraction of hours clipped:", clip_frac)
cf = np.mean(pv_mw) / PV_NAMEPLATE_MWp
print("PV capacity factor:", cf)
print("A_PV (m2):", A_PV)
print("A_PV_unit (m2/kWp):", A_PV_unit)
print("installed_pv_kwp:", installed_pv_kwp)
print("eta_pv min/mean/max:", float(np.min(eta_pv)), float(np.mean(eta_pv)), float(np.max(eta_pv)))
print("G_beta min/mean/max:", float(np.min(G_beta)), float(np.mean(G_beta)), float(np.max(G_beta)))
print("PV nameplate (MWp):", PV_NAMEPLATE_MWp)
print("pv_mw max:", float(np.max(pv_mw)))
print("pv_mw_raw max:", float(np.max(pv_mw_raw)))

print("Model-ready inputs:", OUT_DIR)
